In this notebook, we will elucidate various methodologies for designing and analyzing experiments, specifically A/B tests, using panel data, employing an illustrative example. 

Panel data typically comprises a collection of time series data pertaining to a particular metric (e.g., yearly revenue) across various units (e.g., states). Moreover, this data often undergoes modifications, commonly referred to as treatments (e.g., a promotional campaign for a subscription service). Our primary objective is to draw useful conclusions from the data.

Our overarching aim is to exemplify two core tasks:

1. **Experimental Design**: In this context, we seek to allocate available units into treatment and control groups. The treated group of units receives a specific treatment, while the control group remains untreated.

2. **Inference on Observational Data**: Here, treatment assignments are already provided, and our goal is to draw conclusions, such as the effectiveness or ineffectiveness of the treatment, from the available data, often referred to as observational data.

To begin, we will initiate by loading the "paneldata" package.

In [1]:
import sys
sys.path.insert(0, '..')
from panel_exp.panel_data import long_df_to_paneldataset, PanelDataset, TimePeriod

/Users/ppavuluri/Desktop/panel_exp/panel_exp/examples/../panel_exp/panel_data.py:527: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert (
/Users/ppavuluri/Desktop/panel_exp/panel_exp/examples/../panel_exp/panel_data.py:536: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert (


We are importing historical quarterly GDP data for various states in the United States into a pandas dataframe [AugSynth](#ben2021augmented).

The dataframe consists of rows, each with the following columns:

- `lngdp`: This column contains the natural logarithm of the state's GDP revenue for a specific quarter.

- `year_qtr`: This column represents the year and corresponding quarter in a numerical format. For example, "1990.25" corresponds to the second quarter of the year 1990.

- `state`: This column specifies the U.S. state associated with the data entry.

### References

- [AugSynth](#ben2021augmented): Ben-Michael, E., Feller, A., Rothstein, J. (2021). The augmented synthetic control method. Journal of the American Statistical Association.

In [2]:
import pandas as pd 
import numpy as np

long_df = pd.read_csv('kansas_parsed.csv', index_col=0)
long_df.iloc[:,0] = long_df.iloc[:,0].apply(lambda x: np.exp(x))
long_df.head()

,lngdp,treated,fips,year_qtr,state
1,71610.00,0,1,1990.00,Alabama
2,72718.25,0,1,1990.25,Alabama
3,73826.50,0,1,1990.50,Alabama
4,74934.75,0,1,1990.75,Alabama
5,76043.00,0,1,1991.00,Alabama


In [3]:
# Remove the treated unit
# In our working example, we treat the state of "Kansas" as the treated unit, while the other states are treated as control units. 
# long_df = long_df[long_df.state != "Kansas"]

We are presently transforming the pandas dataframe into a paneldata object. A paneldata object encompasses the following components:

- `wide_data`: This is a pandas dataframe structured as `n_units` (number of units) by `n_time` (number of time periods). Each row corresponds to a specific unit (e.g., a state), and each column represents a particular time period. This dataframe contains the actual data observations.

- `treated_periods`: This is a list that contains all of the time periods during which a treatment or intervention was applied. For instance, if the dataframe includes data for the year 2000, "2000.00" from the provided paneldata, it would be included in this list.

- `treated_units`: This is a list that contains all of the units (e.g., states) that have undergone a treatment or intervention. For example, if the state of Kansas received treatment, it would be listed in this component.

In [4]:
# Create a panel dataset
panel_data = long_df_to_paneldataset(long_df, "year_qtr", "state", "lngdp", ["Kansas"], 2000)
print(panel_data.summarize())


        Panel Dataset Summary
        ---------------------
        Number of time points: 105
        Number of units: 50
        Number of treated units: 1
        Treated units: ['Kansas']
        Treated periods: [TimePeriod(start=2000, end=None)]
        


In [5]:
panel_data.wide_data

# convert a dataframe to a numpy matrix
panel_data.wide_data.values

(panel_data.wide_data.values).shape

panel_data.wide_data


year_qtr,1990.00,1990.25,1990.50,1990.75,1991.00,1991.25,1991.50,1991.75,1992.00,1992.25,...,2013.75,2014.00,2014.25,2014.50,2014.75,2015.00,2015.25,2015.50,2015.75,2016.00
state,,,,,,,,,,,,,,,,,,,,,
Alabama,11.178990,11.194348,11.209473,11.224373,11.239054,11.256060,11.272782,11.289229,11.305409,11.315520,...,12.163572,12.154069,12.170471,12.188748,12.188642,12.194218,12.206887,12.215731,12.215716,12.220345
Alaska,10.128230,10.100318,10.071605,10.042042,10.011579,10.016839,10.022070,10.027275,10.032452,10.038270,...,10.984377,10.979564,10.981404,10.974883,10.950105,10.892768,10.907368,10.881795,10.852865,10.820598
Arizona,11.165239,11.174841,11.184352,11.193773,11.203107,11.234388,11.264720,11.294160,11.322757,11.342570,...,12.525355,12.530508,12.538426,12.556251,12.559402,12.574943,12.583954,12.588235,12.603969,12.608035
Arkansas,10.563078,10.581597,10.599780,10.617638,10.635182,10.655693,10.675792,10.695495,10.714818,10.728868,...,11.654850,11.663042,11.676642,11.684481,11.691666,11.676251,11.689061,11.696871,11.695947,11.695297
California,13.558629,13.563976,13.569294,13.574584,13.579846,13.585310,13.590743,13.596147,13.601522,13.607416,...,14.643806,14.648005,14.666093,14.686874,14.693069,14.715586,14.732883,14.739666,14.748138,14.759022
Colorado,11.232828,11.246610,11.260205,11.273618,11.286853,11.310197,11.333009,11.355312,11.377129,11.400778,...,12.590719,12.600275,12.621359,12.642216,12.655819,12.650778,12.661061,12.665139,12.666812,12.659985
Connecticut,11.514614,11.517868,11.521112,11.524345,11.527568,11.538067,11.548457,11.558740,11.568918,11.573449,...,12.406005,12.394525,12.400463,12.413091,12.421668,12.437823,12.454954,12.452155,12.459117,12.464155
Delaware,9.899881,9.922432,9.944486,9.966063,9.987185,10.000444,10.013530,10.026446,10.039198,10.045746,...,11.060227,11.067060,11.094102,11.115161,11.119453,11.141296,11.156779,11.145839,11.157007,11.146647
Florida,12.455231,12.465968,12.476592,12.487104,12.497506,12.513263,12.528776,12.544051,12.559096,12.576275,...,13.607421,13.611044,13.626679,13.642127,13.655397,13.675450,13.688342,13.705091,13.721159,13.717683



In this section, we delve into the discussion of several randomization strategies. Within each of these strategies, we employ random assignment to allocate units from the panel data into either "Treatment" or "Control" groups. Such randomization strategies build robustness into our models to handle uncertainity in the potential outcomes and reduces the risk of confounding.

By employing a variety of randomization strategies, we distribute different states (considered as pre-treatment units) into two distinct groups. For instance, in one randomization, Massachusetts may be assigned to the "Treatment" group, while Alabama may be assigned to the "Control" group. 

In [4]:
from panel_exp.design import CompleteRandomization, ThinningDesign, Rerandomization, QuickBlock, greedy_match_markets
from panel_exp.design.design_metrics import imbalance

**Complete Randomization**: In this approach, we implement a fully random allocation strategy. A fixed proportion of the units, such as 50% (e.g., denoted as 0.5), is chosen entirely at random and assigned to the "Treatment" group, while the remaining half is assigned to the "Control" group.

The outcome of this assignment process is a paneldata object. After this assignment, we can conveniently access the units designated as treated units using the "treated_units" field, as shown below:

In [8]:
panel_data = long_df_to_paneldataset(long_df, "year_qtr", "state", "lngdp")

cr = CompleteRandomization(treatment_probability=0.5)
paneldata_post_assignment = cr.assign(panel_data)
paneldata_post_assignment.treated_units

We measure the performance of randomization strategy using the imbalance between the treatment and control groups -- higher imbalance correlates with worse generalization performance, i.e., low level of robustness. This helps to make the groups more comparable and reduces the risk of confounding, which can lead to incorrect causal inferences.

Let $x_i \in \mathbb{R}^d$ denote the covariates of unit $i \in \{1, 2, \cdots, n\}$. Given a treatment assignment $z \in \{0, 1\}^n$ indicating the assignment of $n$ units to treatment or control groups, imbalance is measured using the following function:

$$ \text{imbalance}(z) = \frac{1}{d} \left\lVert \sum_{i=1}^n \frac{x_i \mathbb{1} \{ i \text{ is assigned treatment} \}}{\sum_{j = 1}^n \mathbb{1} \{ j \text{ is assigned treatment} \}} - \sum_{i=1}^n \frac{x_i \mathbb{1} \{ i\text{ is assigned control} \}}{\sum_{j = 1}^n \mathbb{1} \{ j\text{ is assigned control} \} } \right\rVert^2 $$

It can also be written as:
$$ \text{imbalance}(z) = \frac{1}{d} \left\lVert \frac{\sum_{i=1}^n x_i z_i}{\sum_{j=1}^n z_j} -  \frac{\sum_{i=1}^n x_i (1-z_i)}{\sum_{j=1}^n (1-z_j)} \right\rVert^2 $$



In [9]:
imbalance(paneldata_post_assignment)

0.10683239395691718

In [10]:
# # We should have 49 units here after dropping Kansas
# # no treated units and no treated periods
# x = TimePeriod()
# x.end  = 2015.25
# ppd = panel_data.drop_period(x)
# ppd.wide_data
# panel_data.summarize

**ThinningDesign**: In this design, the main idea is to partition the units into two groups such that the imbalance function with $l_{\infty}$ error is minimized, instead of the $l_2$ error as defined in the works of [Thinning](#dwivedi2021), [OnlineDesign](#arbour2022).

$$ \text{imbalance}(z) =  \max \left( \frac{\sum_{i=1}^n x_i z_i}{\sum_{j=1}^n z_j} -  \frac{\sum_{i=1}^n x_i (1-z_i)}{\sum_{j=1}^n (1-z_j)} \right) $$


### References
- [Thinning](#dwivedi2021): Dwivedi, R., & Mackey, L. (2021). Kernel Thinning. In Conference on Learning Theory (pp. 1753-1753). PMLR.
- [OnlineDesign] (#arbour2022) :Arbour, D., Dimmery, D., Mai, T., & Rao, A. (2022). Online Balanced Experimental Design. In International Conference on Machine Learning (pp. 844-864). PMLR.

In [11]:
dm = ThinningDesign(treatment_probability=0.5)
imbalance(cr.assign(panel_data))

0.060949883931042566

**Rerandomization**: We repeat the process of randomization (e.g., using CompleteRandomization or ThinningDesign) a fixed number of times and pick the randomized assignment with lowest imbalance.

We generate the treatment vector $z_t \in \{ 0, 1 \}^n$ for the $n$ units repeatedly over $T$ times, i.e., $t \in \{1, 2, \cdots, T\}$. The treatment vectors $z_t$ can be generated through any of the previously discussed randomized strategies as shown below using the argument `base_randomizer`. Finally, we pick the vector $z_t$ that obtains the minimum imbalance:

$$z^* = \arg \min_{t \in \{1, 2, \cdots, T\}} \text{imbalance}(z_t)$$

In [12]:
rerand_dm = Rerandomization(
    treatment_probability=0.5, 
    target_imbalance=1e-4, 
    max_iter=10_000, 
    base_randomizer_cls=CompleteRandomization
)
imbalance(rerand_dm.assign(panel_data))

rerand_dm = Rerandomization(
    treatment_probability=0.5, 
    target_imbalance=1e-4, 
    max_iter=10_000, 
    base_randomizer_cls=ThinningDesign
)
imbalance(rerand_dm.assign(panel_data))

9.923028741432388e-05

**Quickblock**: It involves creating subgroups within a given dataset based on available units to ensure that treatment and control groups have similar distributions of these units [Quickblock](#higgins2016). 

### References

- [Quickblock](#higgins2016): Higgins, M. J., Sävje, F., & Sekhon, J. S. (2016). Improving massive experiments with threshold blocking. Proceedings of the National Academy of Sciences, 113(27), 7369-7376.


In [13]:
qb = QuickBlock(treatment_probability=0.5)
imbalance(qb.assign(panel_data))

/opt/homebrew/lib/python3.10/site-packages/sklearn/neighbors/_distance_metric.py:10: FutureWarning: sklearn.neighbors.DistanceMetric has been moved to sklearn.metrics.DistanceMetric in 1.0. This import path will be removed in 1.3
  warnings.warn(


0.019437637472915256

In this section, we show the performance of the synthetic control method called AugSynth. AugSynth expresses panel data associated with a fixed treated unit (e.g., Kansas) with respect to the remaining units (e.g., other states) using a linear model. 



In [43]:
# !pip3 install cmake

In [44]:
# !pip3 install osqp==0.6.1

In [45]:
# !pip3 install cvxpy 

In [46]:
from __future__ import annotations
import numpy as np
import pandas as pd
import scipy.stats as st
from dataclasses import dataclass

from matplotlib import pyplot as plt
from typing import Dict, Optional
from abc import (
  ABC,
  abstractmethod,
)

from panel_exp.panel_data import long_df_to_paneldataset
from panel_exp.methods.tbr import TBR
from panel_exp.methods.scm import SyntheticControl, AugSynth



Here are additional details:

**Data**:
- The dataset contains time-series data for multiple states.
- Recall that the main metric of interest is denoted by "lngdp", GDP of the particular state in a given quarter.
- The dataset spans multiple years, covering a defined observation period.

**Treatment**:
- In the year 2000, a treatment or intervention was implemented.
- This treatment was specifically applied to the state of "Kansas."

**Plot**:
- The analysis involves the creation of two curves: the "Observed" curve and the "Inferred" curve.
- Both curves are plotted on the same graph.
- The y-axis of the graph represents the variable "lngdp."
- The x-axis is normalized, meaning it is scaled in a way that allows for a uniform comparison across time.
- The treatment year, 2000, is indicated on the x-axis.
- The statement "the treatment at '2000' roughly corresponds to '40%' of total duration" suggests that the treatment duration, starting in 2000, constitutes approximately 40% of the entire observation period.

This analysis appears to involve evaluating the impact of the treatment on the economic output of Kansas compared to other states, with both observed and inferred curves illustrating the trends in "lngdp" over time.

The first plot shows both the cuves on the same figure. The second plot shows the difference between the curves vs time, while the third plot shows the **cumulative** difference vs time.

In [15]:
panel_data = long_df_to_paneldataset(long_df, "year_qtr", "state", "lngdp")

cr = CompleteRandomization(treatment_probability=0.5)
td = ThinningDesign(treatment_probability=0.5)
gm = greedy_match_markets()

design = [cr,td,gm][2]

test_whitelist = ['Kansas']

#to make test whitelist strictly inclusive, you must pass test_blacklist in the below manner (i.e. include all dmas not part of test_whitelist)
test_blacklist = [geo for geo in panel_data.units.tolist() if geo not in test_whitelist]

paneldata_post_assignment = design.assign(panel_data=panel_data,test_whitelist=test_whitelist,test_blacklist=test_blacklist)


running iteration 1
running iteration 2
running iteration 3
running iteration 4
running iteration 5
running iteration 6
running iteration 7
running iteration 8
running iteration 9
running iteration 10
running iteration 11
running iteration 12
running iteration 13
running iteration 14
running iteration 15
running iteration 16
running iteration 17
running iteration 18
running iteration 19
running iteration 20


/Users/ppavuluri/Desktop/panel_exp/panel_exp/examples/../panel_exp/panel_data.py:515: FutureWarning: In a future version, the Index constructor will not infer numeric dtypes when passed object-dtype sequences (matching Series behavior)
  df = df.pivot(columns=time_column, values=value_column)


In [16]:
print('Test group: %s \n\nControl group: %s' %(paneldata_post_assignment.treated_units, [unit for unit in paneldata_post_assignment.units if unit not in paneldata_post_assignment.treated_units]))


Test group: ['Kansas'] 

Control group: ['Alaska', 'Idaho', 'Mississippi', 'Nebraska', 'New Mexico', 'North Dakota', 'Oklahoma', 'Oregon', 'Pennsylvania', 'South Carolina', 'West Virginia', 'Wyoming']


In [49]:
test = paneldata_post_assignment.treated_units
control = [unit for unit in paneldata_post_assignment.units if unit not in paneldata_post_assignment.treated_units]

trimmed_long_df = long_df[long_df.state.isin(test+control)]

panel_data = long_df_to_paneldataset(trimmed_long_df, "year_qtr", "state", "lngdp", test, [2000])
asynth = AugSynth()
synth = SyntheticControl()
synth.run_analysis(panel_data)
synth.plot()

SolverError: Either candidate conic solvers (['SCIPY']) do not support the cones output by the problem (SOC, NonNeg, Zero), or there are not enough constraints in the problem.

Matching Design
Expose ACS (Lupinov)
Uncertainity Estimation Conformal Inference
